In [3]:
import pandas as pd
import pickle
import importlib
import sys
import os
from syllabification import tokenize, syllabify_sentences
from markov_models import MarkovModel 
from helpers import get_word_data, get_info_rate, update_values_in_csv
import warnings
import re
import numpy as np

# define parameters
language = "DEU"
input_type = "words"

language_paths_words = {
    "FRA": "Z:/data/FRA/Lexique383.tsv",
    "JPN": "Z:/data/JPN/jpn.txt",
    "CMN": "Z:/data/CMN/cmn.txt",
    "VIE": "Z:/data/VIE/vie.txt",
    "YUE": "Z:/data/YUE/yue.txt", 
    "ENG": "Z:/data/ENG/eng.txt",
    "DEU": "Z:/data/DEU/WebCelex_German.xlsx",
}

In [4]:
n_values = [1, 2, 3, 4]  # For bigram, trigram, and 4-gram models
markov_models = {}

for n in n_values:
    
    # Create and build the Markov model
    model = MarkovModel(n)

    if input_type == "sentences": 
        # Load the paired data
        with open(f"produced_data/{language}/sentence_pairs.pkl", "rb") as f: 
            sentence_pairs = pickle.load(f)

        # Merge all transcribed (syllabified) sentences into one list
        merged_sentences = []
        for tokenized, transcribed in sentence_pairs:
            merged_sentences.extend(transcribed)

        model.build(merged_sentences, input_type)

    elif input_type == "words": 
        try:
            path = language_paths_words[language]
        except KeyError:
            raise ValueError(f"Unsupported language: {language}")

        # Load the data
        words = get_word_data(path)
       
        print(f"\nTraining a Markov Model with n = {n}:")
        # Build the markov model
        model.build(words, input_type)


    # Compute the conditional entropy (information density)
    info_density = model.compute_conditional_entropy()
    print(f"Information Density: {info_density:.4f}")

    # Compute the information rate (bits per second)
    info_rate = get_info_rate(info_density, language)
    print(f"Information Rate: {np.mean(info_rate):.4f}")
    
    # Update the CSV file with the computed info_density and info_rate
    update_values_in_csv(language, info_density, n, 'ID')
    update_values_in_csv(language, info_rate, n, 'IR')

    # Store model for later use 
    markov_models[n] = model

    # Display exactly 3 examples
    example_count = 0
    print("\nExample probabilities (p(x, y)):")

    for (prefix, suffix), p_xy in model.cond_probs.items():
        print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
        example_count += 1
        if example_count == 3:
            break
    
    # Save the model to a file
    model.save_model(language, input_type)

Language: DEU


c:\Users\emill\anaconda3\envs\coupe_env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



Training a Markov Model with n = 1:
ngram examples: [('a',), ('a',), ('a',), ('a',), ('a',)]
Information Density: 8.8205
Information Rate: 42.4558
Updated ID_unigram_esidaine
Updated IR_unigram_esidaine

Example probabilities (p(x, y)):
p(() -> a) = 0.0037
p(() -> l@) = 0.0033
p(() -> al) = 0.0005

✅ Saved 1-gram model to 'produced_data/DEU/'
Language: DEU


c:\Users\emill\anaconda3\envs\coupe_env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



Training a Markov Model with n = 2:
ngram examples: [('a', 'l@'), ('a', 'l@'), ('a', 'l@'), ('a', 'l@'), ('a', 'l@')]
Information Density: 3.8301
Information Rate: 18.4355
Updated ID_bigram_esidaine
Updated IR_bigram_esidaine

Example probabilities (p(x, y)):
p(('a',) -> l@) = 0.0194
p(('a',) -> l@n) = 0.0524
p(('g@',) -> alt) = 0.0013

✅ Saved 2-gram model to 'produced_data/DEU/'
Language: DEU


c:\Users\emill\anaconda3\envs\coupe_env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



Training a Markov Model with n = 3:
ngram examples: [('as', 'gW', '@r'), ('as', 'gW', '@rn'), ('p', 'g@', 'En'), ('g@', 'En', 'd@rt'), ('p', 'g@', 'En')]
Information Density: 1.5171
Information Rate: 7.3024
Updated ID_trigram_esidaine
Updated IR_trigram_esidaine

Example probabilities (p(x, y)):
p(('as', 'gW') -> @r) = 0.5000
p(('as', 'gW') -> @rn) = 0.5000
p(('p', 'g@') -> En) = 0.0019

✅ Saved 3-gram model to 'produced_data/DEU/'
Language: DEU


c:\Users\emill\anaconda3\envs\coupe_env\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)



Training a Markov Model with n = 4:
ngram examples: [('p', 'g@', 'En', 'd@rt'), ('p', 'g@', 'En', 'd@rt'), ('p', 'g@', 'En', 'd@rt'), ('p', 'g@', 'En', 'd@rt'), ('p', 'g@', 'En', 'd@rt')]
Information Density: 0.9264
Information Rate: 4.4589
Updated ID_4gram_esidaine
Updated IR_4gram_esidaine

Example probabilities (p(x, y)):
p(('p', 'g@', 'En') -> d@rt) = 1.0000
p(('p', 'u', 'En') -> d@rn) = 1.0000
p(('p', 'En', 'd@r') -> t@) = 0.0417

✅ Saved 4-gram model to 'produced_data/DEU/'


In [5]:
# prepare transcribed sentences 

if language == "FRA" and input_type == "sentences": 
    path = "Z:/data/FRA/french_sentences.txt"  # Use the mounted drive letter

    # Read each line as a sentence
    tokenized_sentences = []
    transcribed_sentences = []


    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            sentence = line.strip()
            if sentence:
                # Tokenize
                tokenized_sentence = tokenize(sentence)
                print(tokenized_sentence)
                tokenized_sentences.append(tokenized_sentence)
                
                # Syllabify
                transcribed_sentence = syllabify_sentences(tokenized_sentence, language="FRA")
                print(transcribed_sentence)
                if transcribed_sentence: 
                    transcribed_sentences.append(transcribed_sentence)

            # Optional: limit for testing
            if i >= 20:
                break


    # show the entries 
    for i, sentence in enumerate(transcribed_sentences[:20]):
        print(f"Sentence {i+1}: {sentence}")

else: 
    warnings.warn("Warning: The specified language is not available yet")


# Save to .pkl
paired_sentences = list(zip(tokenized_sentences, transcribed_sentences))

with open("produced_data/{language}/preprocessed_{input_type}.pkl", "wb") as f:
    pickle.dump(paired_sentences, f)

print(f"✅ Saved tokenized and transcribed sentences to 'produced_data/{language}'")


C:\Users\emill\AppData\Local\Temp\ipykernel_34572\1726129514.py:36: UserWarning: Warning: The specified language is not available yet
  warnings.warn("Warning: The specified language is not available yet")


NameError: name 'tokenized_sentences' is not defined